# 02 preparacion y logistica


## 3. Preparación y División de Datos


In [ ]:

train["Ratio_Monto_Ingreso"] = (
    train["Monto_Solicitado"] / train["Ingreso_Mensual"]
)

X = train[
    [
        "Monto_Solicitado",
        "Tasa_Interes",
        "Ingreso_Mensual",
        "Ratio_Monto_Ingreso"
    ]
]

y = train["Estado_Final"]


In [ ]:
# SPLIT DATOS / TRAIN, VALIDATION, TEST

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



## 4. Primeros modelos: regresión logística

Empezamos con una regresión logística usando los datos de las solicitudes, sin incorporar todavía el historial bancario. Separamos entrenamiento y validación, calculamos accuracy y revisamos la matriz de confusión. El primer modelo tuvo cerca de 84% de accuracy, pero esa medida no nos decía cuánto dinero podíamos ganar prestando. 

In [ ]:
# BASELINE
# construimos un primer modelo usando solo variables numericas de solicitudes_train.csv, sin comportamiento bancario ni variables cat.
X = train[
    [
        "Monto_Solicitado",
        "Tasa_Interes",
        "Ingreso_Mensual",
        "Ratio_Monto_Ingreso"
    ]
]

y = train["Estado_Final"]

X.head() # para ver las features
y.head() # para ver los resultados




In [ ]:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

print(y_train.value_counts(normalize=True))
print(y_val.value_counts(normalize=True))
#para PROBAR SPLIT, ver q los datos se hayan dividido bien

In [ ]:
# MODELO REGRESION LOGISTICA

from sklearn.linear_model import LogisticRegression

modelo = LogisticRegression(max_iter=1000) #max iter le da hasta 1000 iteraciones para encontrar sus parametros




In [ ]:
# ENTRENAR MODELO
modelo.fit(X_train, y_train) #aca aprende el modelo, que combinacion de variables esta asociada con 0 y 1

In [ ]:
predicciones = modelo.predict(X_val) # hacemos predicciones, una por cada fila de validation

predicciones[:20] #muestra las primeras 20

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_val, predicciones)
accuracy

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_val, predicciones)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_val, predicciones))
# la  baseline obtiene un 84% de accuracy, osea que predice casi todos como pagados
# solo detecta bien 10 defaults de 1034 en validation
# el modelo se esta aprovechando del desbalance en el  dataset, osea en que hay 84% de casos de prestamos pagados


In [ ]:
probabilidades = modelo.predict_proba(X_val)[:, 1]
# para analizar a la regresion logistica y ver si aprendio algo, vemos probabilidades

probabilidades[:20]

In [ ]:
import pandas as pd

analisis_prob = pd.DataFrame({
    "Estado_Real": y_val,
    "Prob_Pago": probabilidades
})

analisis_prob.groupby("Estado_Real")["Prob_Pago"].describe()
#para ver si hay relacion en las probabilidades y el resultado, osea si un prestamo default tiene probabilidad mas baja de ser
#pagado segun el modelo

### Modelo 2 — Regresión logística con clases balanceadas

Probamos class_weight='balanced porque el primer modelo detectaba muy pocos incumplimientos. La exactitud bajó aproximadamente del 84% al 65%, mientras que el recall de default pasó de cerca del 1% al 44%, según las salidas de esta prueba. 

In [ ]:
modelo_balanceado = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

modelo_balanceado.fit(X_train, y_train)

In [ ]:
predicciones_balanceadas = modelo_balanceado.predict(X_val)

In [ ]:
print("Accuracy:", accuracy_score(y_val, predicciones_balanceadas))

print("Matriz de confusión:")
print(confusion_matrix(y_val, predicciones_balanceadas))

print("Reporte:")
print(classification_report(y_val, predicciones_balanceadas))

# vemos que mejora bastante la prediccion de defaults, ahora no deja pasar todos, sino que acierta en 460, muchos mas que antes.
# tenemos un RECALL de clase 0 de 0.44, osae que detecta el 44% de los prestamos que terminaron en default.

# por otro lado la  PRECISION es de 0.22, osea que de todos los que marco como default, solo el 22% terminaron en default..

In [ ]:
# al usar el class_weight="balanced" el accuracy baja de 84% a 65%, pero el recall de la clase default mejora del 1 al 44%
# osea que el modeo balanceado detecta mucho mejor los defualts, lo que nos interesa, aunque a costa de clasififcar
# como riesgoso a gente que pagaban, hay que reducir esto

# para ver que  modelo es mejor hay que ver el impacto economico

### Modelo 3 — Regresión logística con escalado

Aplicamos StandardScaler antes de entrenar para que variables con valores muy diferentes no influyeran por su escala numérica. Después usamos las probabilidades para ordenar préstamos por ganancia esperada y calculamos lo que habría ocurrido en validación. Mantuvimos esta prueba como comparación, sin elegir el modelo únicamente por accuracy.

In [ ]:
# la idea de standard scaler es transformar cada feature aproximadamente a una escala comparable
# no trabaja con datos originales de train sino q genera versiones transformadas para que la regresion
# trabaje con ellas



In [ ]:
# creamos modelo escalado
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

modelo_escalado = Pipeline([
    ("scaler", StandardScaler()),
    ("modelo", LogisticRegression(max_iter=1000))
])

modelo_escalado.fit(X_train, y_train)
# scaler aprende solamente de X_train, no de validation

In [ ]:
predicciones_escaladas = modelo_escalado.predict(X_val)

probabilidades_escaladas = modelo_escalado.predict_proba(X_val)[:, 1]
# probamos

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_val, predicciones_escaladas))
print(confusion_matrix(y_val, predicciones_escaladas))
print(classification_report(y_val, predicciones_escaladas))

In [ ]:
 # nueva cartera de validacion
cartera_val_escalada = train.loc[X_val.index, [
    "ID_Solicitud",
    "Monto_Solicitado",
    "Tasa_Interes",
    "Estado_Final"
]].copy()

cartera_val_escalada["Prob_Pago"] = probabilidades_escaladas
cartera_val_escalada["Prob_Default"] = 1 - cartera_val_escalada["Prob_Pago"]

In [ ]:
# EV
cartera_val_escalada["EV"] = (
    cartera_val_escalada["Prob_Pago"]
    * cartera_val_escalada["Monto_Solicitado"]
    * cartera_val_escalada["Tasa_Interes"]
    -
    cartera_val_escalada["Prob_Default"]
    * cartera_val_escalada["Monto_Solicitado"]
)
# usa la logica de la consigna, si paga ganamos intereses, sino default y perdemos

In [ ]:
# presupuesto de 2 millones
cartera_val_escalada = cartera_val_escalada.sort_values(
    "EV",
    ascending=False
)

presupuesto = 2_000_000
capital_usado_escalado = 0
cartera_val_escalada["Aprobar"] = 0

for indice, fila in cartera_val_escalada.iterrows():

    if fila["EV"] <= 0:
        continue

    if capital_usado_escalado + fila["Monto_Solicitado"] <= presupuesto:
        cartera_val_escalada.loc[indice, "Aprobar"] = 1
        capital_usado_escalado += fila["Monto_Solicitado"]

In [ ]:
print("Capital utilizado:", capital_usado_escalado)
print(
    "Préstamos aprobados:",
    cartera_val_escalada["Aprobar"].sum()
)

In [ ]:
# CALCULO DE GANANCIA REAL DE VALIDATION - usamos la funcion creada, NO ENTREGAR
def ganancia_real_escalada(fila):
    if fila["Aprobar"] == 0:
        return 0

    if fila["Estado_Final"] == 1:
        return fila["Monto_Solicitado"] * fila["Tasa_Interes"]

    return -fila["Monto_Solicitado"]


cartera_val_escalada["Ganancia_Real"] = cartera_val_escalada.apply(
    ganancia_real_escalada,
    axis=1
)

ganancia_escalada = cartera_val_escalada["Ganancia_Real"].sum()

print("Ganancia real simulada:", ganancia_escalada)
# vemos que al usar el StandardScaler nos sigue detectando casi todo como que paga, solo detecto 9 defaults bien
#1025 los mando como pagados
# en validacion el cambio nos llevo a elegir prestamos perores, la ganancia cayo de 347 a 180mil, nos quedamos con el anterior para la sub 1.